### 1. Carga de datos

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, torch, gc

esqueleto = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_panel_corregido.parquet')
print(esqueleto.shape)

_torch_load_original = torch.serialization.load
def _torch_load_patched(*args, **kwargs):
    kwargs['weights_only'] = False
    return _torch_load_original(*args, **kwargs)
torch.load = _torch_load_patched

!pip install pytorch-forecasting pytorch-lightning lightning -q

import pytorch_forecasting.data.encoders as pf_encoders
torch.serialization.add_safe_globals([
    pf_encoders.GroupNormalizer, pf_encoders.TorchNormalizer,
    pf_encoders.NaNLabelEncoder, pf_encoders.EncoderNormalizer, pf_encoders.MultiNormalizer,
])

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer

print('GPU disponible:', torch.cuda.is_available())

Mounted at /content/drive
(4804366, 22)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.2 MB/s eta 0:00:00
GPU disponible: True


### 2. Preparación de variables

In [2]:
if 'type' in esqueleto.columns:
    esqueleto = esqueleto.rename(columns={'type': 'store_type'})
for col in ['store_nbr', 'item_nbr', 'city', 'state', 'store_type', 'cluster', 'family', 'class', 'perishable']:
    esqueleto[col] = esqueleto[col].astype(str)

### 3. Partición de datos

In [3]:
fecha_max = esqueleto['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)

train = esqueleto[esqueleto['date'] < val_inicio]
val = esqueleto[(esqueleto['date'] >= val_inicio) & (esqueleto['date'] < test_inicio)]
print('Train:', train.shape, '| Val:', val.shape)

Train: (4322665, 22) | Val: (238860, 22)


### 4. Baseline naive

In [4]:
esqueleto['pred_naive'] = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(7)

### 5. Ponderación por volumen

In [5]:
volumen_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].transform('mean')
esqueleto['peso_muestra'] = 1 / np.sqrt(volumen_por_serie + 0.1)
esqueleto['peso_muestra'] = esqueleto['peso_muestra'] / esqueleto['peso_muestra'].mean()

### 6. Reconstrucción de TimeSeriesDataSet

In [6]:
max_encoder_length, max_prediction_length = 90, 30
training_cutoff = train['time_idx'].max()

training = TimeSeriesDataSet(
    esqueleto[esqueleto.time_idx <= training_cutoff],
    time_idx="time_idx", target="unit_sales", group_ids=["store_nbr", "item_nbr"],
    max_encoder_length=max_encoder_length, max_prediction_length=max_prediction_length,
    static_categoricals=["city", "state", "store_type", "cluster", "family", "class", "perishable"],
    time_varying_known_reals=["time_idx", "year", "month", "day_of_week", "is_weekend",
                               "es_feriado", "onpromotion", "edad"],
    time_varying_unknown_reals=["unit_sales", "dcoilwtico"],
    target_normalizer=GroupNormalizer(groups=["store_nbr", "item_nbr"], transformation="log1p"),
    weight="peso_muestra", add_relative_time_idx=True, add_target_scales=True,
    add_encoder_length=True, allow_missing_timesteps=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training, esqueleto, min_prediction_idx=training_cutoff + 1, stop_randomization=True
)

/usr/local/lib/python3.13/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 88 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_nbr': '1', '__group_id__item_nbr': '1428779'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2002136'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027777'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027827'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053610'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053614'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2033805'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1

### 7. Dataloaders

In [7]:
batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

### 8. Escala de naive (periodo de train) para MASE/RMSSE

In [8]:
train_naive = train.copy()
train_naive['pred_naive'] = train_naive.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(7)
train_naive = train_naive.dropna(subset=['pred_naive'])
train_naive['err_abs'] = (train_naive['unit_sales'] - train_naive['pred_naive']).abs()
train_naive['err_sq'] = (train_naive['unit_sales'] - train_naive['pred_naive']) ** 2

escala_series = train_naive.groupby(['store_nbr', 'item_nbr']).agg(
    escala_mae=('err_abs', 'mean'),
    escala_rmse=('err_sq', lambda x: x.mean() ** 0.5)
).reset_index()

### 9. Evaluación de modelos

In [9]:
def evaluar_modelo_completo(modelo, esqueleto_eval, test_cutoff, training, nombre_col, n_chunks=8, batch_size=128):
    series_unicas = esqueleto_eval[['store_nbr', 'item_nbr']].drop_duplicates().reset_index(drop=True)
    chunks = np.array_split(series_unicas, n_chunks)

    resultados_parciales = []
    quantiles = None

    for i, chunk in enumerate(chunks):
        idx_chunk = pd.MultiIndex.from_frame(chunk)
        idx_esqueleto = pd.MultiIndex.from_frame(esqueleto_eval[['store_nbr', 'item_nbr']])
        esqueleto_chunk = esqueleto_eval[idx_esqueleto.isin(idx_chunk)]

        testing_chunk = TimeSeriesDataSet.from_dataset(
            training, esqueleto_chunk, min_prediction_idx=test_cutoff + 1, stop_randomization=True
        )
        dataloader_chunk = testing_chunk.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

        predictions_q = modelo.predict(
            dataloader_chunk, mode="quantiles", return_y=True, return_index=True,
            trainer_kwargs=dict(accelerator="auto", logger=False)
        )
        quantiles = modelo.loss.quantiles

        y_true = predictions_q.y[0].cpu().numpy()
        y_pred_todos = predictions_q.output.cpu().numpy()
        indice = predictions_q.index
        n_ventanas, horizonte = y_true.shape

        filas = {
            'store_nbr': np.repeat(indice['store_nbr'].values, horizonte),
            'item_nbr': np.repeat(indice['item_nbr'].values, horizonte),
            'time_idx': np.repeat(indice['time_idx'].values, horizonte) + np.tile(np.arange(horizonte), n_ventanas),
            'actual': y_true.flatten(),
        }
        for j, q in enumerate(quantiles):
            filas[f'{nombre_col}_q{q}'] = y_pred_todos[..., j].flatten()

        resultado_chunk = pd.DataFrame(filas).drop_duplicates(subset=['store_nbr', 'item_nbr', 'time_idx'], keep='first')
        resultados_parciales.append(resultado_chunk)
        print(f'Chunk {i+1}/{n_chunks}: {len(chunk)} series, {resultado_chunk.shape[0]} filas')

        del predictions_q, y_true, y_pred_todos, indice, testing_chunk, dataloader_chunk, resultado_chunk
        gc.collect()
        torch.cuda.empty_cache()

    resultados = pd.concat(resultados_parciales, ignore_index=True)
    print(f'{nombre_col}: {resultados.shape[0]} filas totales (esperado cercano a 242.708)')
    return resultados, quantiles

In [10]:
#Evaluación t1
test_cutoff = val['time_idx'].max()
tft_t1 = TemporalFusionTransformer.load_from_checkpoint('/content/drive/MyDrive/TFM/tft_t1_v2.ckpt')
comp_t1, quantiles = evaluar_modelo_completo(tft_t1, esqueleto, test_cutoff, training, 't1', n_chunks=16)
comp_t1.to_parquet('/content/drive/MyDrive/TFM/pred_tft_t1_v2.parquet', index=False)
del tft_t1, comp_t1
gc.collect(); torch.cuda.empty_cache()
print('t1 evaluado y guardado')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU avail

Chunk 1/16: 249 series, 15177 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 2/16: 249 series, 15164 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

...

Chunk 14/16: 248 series, 15128 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 15/16: 248 series, 15128 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 16/16: 248 series, 15128 filas
t1: 242708 filas totales (esperado cercano a 242.708)
t1 evaluado y guardado


In [11]:
#Evaluación t2
tft_t2 = TemporalFusionTransformer.load_from_checkpoint('/content/drive/MyDrive/TFM/tft_t2_v2.ckpt')
comp_t2, quantiles = evaluar_modelo_completo(tft_t2, esqueleto, test_cutoff, training, 't2', n_chunks=16)
comp_t2.to_parquet('/content/drive/MyDrive/TFM/pred_tft_t2_v2.parquet', index=False)
del tft_t2, comp_t2
gc.collect(); torch.cuda.empty_cache()
print('t2 evaluado y guardado')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU avail

Chunk 1/16: 249 series, 15177 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 2/16: 249 series, 15164 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

...

Chunk 14/16: 248 series, 15128 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 15/16: 248 series, 15128 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 16/16: 248 series, 15128 filas
t2: 242708 filas totales (esperado cercano a 242.708)
t2 evaluado y guardado


In [12]:
#Evaluación t3
tft_t3 = TemporalFusionTransformer.load_from_checkpoint('/content/drive/MyDrive/TFM/tft_t3_v2.ckpt')
comp_t3, quantiles = evaluar_modelo_completo(tft_t3, esqueleto, test_cutoff, training, 't3', n_chunks=16)
comp_t3.to_parquet('/content/drive/MyDrive/TFM/pred_tft_t3_v2.parquet', index=False)
del tft_t3, comp_t3
gc.collect(); torch.cuda.empty_cache()
print('t3 evaluado y guardado')

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU avail

Chunk 1/16: 249 series, 15177 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 2/16: 249 series, 15164 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

...

Chunk 14/16: 248 series, 15128 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 15/16: 248 series, 15128 filas


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Chunk 16/16: 248 series, 15128 filas
t3: 242708 filas totales (esperado cercano a 242.708)
t3 evaluado y guardado


In [13]:
#Unificar
pred_t1 = pd.read_parquet('/content/drive/MyDrive/TFM/pred_tft_t1_v2.parquet')
pred_t2 = pd.read_parquet('/content/drive/MyDrive/TFM/pred_tft_t2_v2.parquet')
pred_t3 = pd.read_parquet('/content/drive/MyDrive/TFM/pred_tft_t3_v2.parquet')

comparacion_tft = pred_t1.merge(pred_t2.drop(columns='actual'), on=['store_nbr', 'item_nbr', 'time_idx']) \
    .merge(pred_t3.drop(columns='actual'), on=['store_nbr', 'item_nbr', 'time_idx']) \
    .merge(esqueleto[['store_nbr', 'item_nbr', 'time_idx', 'pred_naive']], on=['store_nbr', 'item_nbr', 'time_idx'], how='left')

comparacion_tft.to_parquet('/content/drive/MyDrive/TFM/comparacion_completa_tft_v2.parquet', index=False)
comparacion_tft.shape

(242708, 26)

### 10. Métricas

In [14]:
def pinball_loss(actual, pred, q):
    e = actual - pred
    return np.maximum(q * e, (q - 1) * e).mean()

def metricas_completas_tft(comp, prefijo, quantiles, escala_series):
    med = comp[f'{prefijo}_q0.5']
    mae = (comp['actual'] - med).abs().mean()
    rmse = ((comp['actual'] - med) ** 2).mean() ** 0.5
    wape = (comp['actual'] - med).abs().sum() / comp['actual'].sum()

    pinballs = [pinball_loss(comp['actual'], comp[f'{prefijo}_q{q}'], q) for q in quantiles]
    pinball_prom = np.mean(pinballs)

    cobertura_80 = ((comp['actual'] >= comp[f'{prefijo}_q0.1']) & (comp['actual'] <= comp[f'{prefijo}_q0.9'])).mean()
    cobertura_96 = ((comp['actual'] >= comp[f'{prefijo}_q0.02']) & (comp['actual'] <= comp[f'{prefijo}_q0.98'])).mean()

    c = comp.merge(escala_series, on=['store_nbr', 'item_nbr'], how='left').copy()
    c['err_abs'] = (c['actual'] - med).abs()
    c['err_sq'] = (c['actual'] - med) ** 2
    por_serie = c.groupby(['store_nbr', 'item_nbr']).agg(
        mae_serie=('err_abs', 'mean'), rmse_serie=('err_sq', lambda x: x.mean() ** 0.5),
        escala_mae=('escala_mae', 'first'), escala_rmse=('escala_rmse', 'first'),
    )
    por_serie['mase'] = por_serie['mae_serie'] / por_serie['escala_mae']
    por_serie['rmsse'] = por_serie['rmse_serie'] / por_serie['escala_rmse']
    por_serie = por_serie.replace([np.inf, -np.inf], np.nan)
    n_excluidas = por_serie['mase'].isna().sum()

    cols_q = [f'{prefijo}_q{q}' for q in quantiles]
    diffs = np.diff(comp[cols_q].values, axis=1)
    prop_cruce = (diffs < 0).any(axis=1).mean()

    return {
        'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'pinball_loss': pinball_prom,
        'cobertura_80% (nominal 80%)': cobertura_80, 'cobertura_96% (nominal 96%)': cobertura_96,
        'MASE_media': por_serie['mase'].mean(), 'MASE_mediana': por_serie['mase'].median(),
        'RMSSE_media': por_serie['rmsse'].mean(), 'RMSSE_mediana': por_serie['rmsse'].median(),
        'prop_cruce_cuantiles': prop_cruce,
        'series_excluidas_escala_naive_cero': n_excluidas,
    }

resultados_tft = pd.DataFrame([
    {'modelo': 'TFT t1', 'configuracion': 'hidden_size=30, batches=200', **metricas_completas_tft(comparacion_tft, 't1', quantiles, escala_series)},
    {'modelo': 'TFT t2', 'configuracion': 'hidden_size=64, batches=200', **metricas_completas_tft(comparacion_tft, 't2', quantiles, escala_series)},
    {'modelo': 'TFT t3', 'configuracion': 'hidden_size=30, batches=400', **metricas_completas_tft(comparacion_tft, 't3', quantiles, escala_series)},
])
resultados_tft.to_csv('/content/drive/MyDrive/TFM/resultados_finales_tft_v2.csv', index=False)
resultados_tft

,modelo,configuracion,MAE,RMSE,WAPE,pinball_loss,cobertura_80% (nominal 80%),cobertura_96% (nominal 96%),MASE_media,MASE_mediana,RMSSE_media,RMSSE_mediana,prop_cruce_cuantiles,series_excluidas_escala_naive_cero
0,TFT t1,"hidden_size=30, batches=200",2.716249,6.232767,0.443305,0.769005,0.683208,0.928433,2.239439,0.583318,0.751587,0.589127,0.0,2
1,TFT t2,"hidden_size=64, batches=200",2.830012,6.278166,0.461872,0.803791,0.686953,0.899307,1.881413,0.620179,0.740181,0.603286,0.0,2
2,TFT t3,"hidden_size=30, batches=400",2.792821,6.359457,0.455802,0.788112,0.720973,0.920658,1.836707,0.599929,0.737192,0.590131,0.0,2


### 11. Test Diebold-Mariano

In [15]:
from scipy import stats

def test_dm(comp, col_referencia, col_modelo, tipo_perdida='L2'):
    d = comp.dropna(subset=[col_referencia, col_modelo, 'actual']).copy()
    if tipo_perdida == 'L2':
        d['err_ref'] = (d['actual'] - d[col_referencia]) ** 2
        d['err_modelo'] = (d['actual'] - d[col_modelo]) ** 2
    else:
        d['err_ref'] = (d['actual'] - d[col_referencia]).abs()
        d['err_modelo'] = (d['actual'] - d[col_modelo]).abs()
    d['diferencia'] = d['err_ref'] - d['err_modelo']
    dif_por_serie = d.groupby(['store_nbr', 'item_nbr'])['diferencia'].mean()
    n = len(dif_por_serie)
    dm_stat = dif_por_serie.mean() / (dif_por_serie.std(ddof=1) / np.sqrt(n))
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return n, dm_stat, p_value

print('--- TFT vs. baseline naive ---')
for modelo, col in [('t1', 't1_q0.5'), ('t2', 't2_q0.5'), ('t3', 't3_q0.5')]:
    for perdida in ['L2', 'L1']:
        n, dm_stat, p_value = test_dm(comparacion_tft, 'pred_naive', col, perdida)
        print(f'TFT {modelo} vs. baseline ({perdida}) -> series: {n} | DM: {dm_stat:.4f} | p-valor: {p_value:.4g}')

--- TFT vs. baseline naive ---
TFT t1 vs. baseline (L2) -> series: 3981 | DM: -0.0440 | p-valor: 0.9649
TFT t1 vs. baseline (L1) -> series: 3981 | DM: 17.3251 | p-valor: 0
TFT t2 vs. baseline (L2) -> series: 3981 | DM: -0.1830 | p-valor: 0.8548
TFT t2 vs. baseline (L1) -> series: 3981 | DM: 14.3987 | p-valor: 0
TFT t3 vs. baseline (L2) -> series: 3981 | DM: -0.3951 | p-valor: 0.6927
TFT t3 vs. baseline (L1) -> series: 3981 | DM: 13.8587 | p-valor: 0


In [16]:
#Test Diebold-Mariano: TFT vs. mejor DeepAR (v9)
comparacion_deepar = pd.read_parquet('/content/drive/MyDrive/TFM/comparacion_completa_v2.parquet')
comparacion_cruzada = comparacion_tft.merge(
    comparacion_deepar[['store_nbr', 'item_nbr', 'time_idx', 'v9_q0.5']],
    on=['store_nbr', 'item_nbr', 'time_idx'], how='inner'
)
print('Filas cruzadas TFT/DeepAR v9:', comparacion_cruzada.shape[0], '(esperado cercano a 242.708)')

print('\n--- TFT vs. DeepAR v9 ---')
for modelo, col in [('t1', 't1_q0.5'), ('t2', 't2_q0.5'), ('t3', 't3_q0.5')]:
    for perdida in ['L2', 'L1']:
        n, dm_stat, p_value = test_dm(comparacion_cruzada, 'v9_q0.5', col, perdida)
        print(f'TFT {modelo} vs. DeepAR v9 ({perdida}) -> series: {n} | DM: {dm_stat:.4f} | p-valor: {p_value:.4g}')

Filas cruzadas TFT/DeepAR v9: 242708 (esperado cercano a 242.708)

--- TFT vs. DeepAR v9 ---
TFT t1 vs. DeepAR v9 (L2) -> series: 3981 | DM: -3.7759 | p-valor: 0.0001594
TFT t1 vs. DeepAR v9 (L1) -> series: 3981 | DM: -5.9841 | p-valor: 2.176e-09
TFT t2 vs. DeepAR v9 (L2) -> series: 3981 | DM: -3.8366 | p-valor: 0.0001248
TFT t2 vs. DeepAR v9 (L1) -> series: 3981 | DM: -9.1180 | p-valor: 0
TFT t3 vs. DeepAR v9 (L2) -> series: 3981 | DM: -3.6254 | p-valor: 0.0002885
TFT t3 vs. DeepAR v9 (L1) -> series: 3981 | DM: -7.1811 | p-valor: 6.914e-13


In [17]:
#Revisar curvas
import pandas as pd, glob

def revisar_curva(nombre_modelo, carpeta='/content/drive/MyDrive/TFM/lightning_logs'):
    archivos = sorted(glob.glob(f'{carpeta}/{nombre_modelo}/version_*/metrics.csv'))
    print(f'{nombre_modelo}: {len(archivos)} fragmento(s) encontrados')
    if not archivos:
        print('  No se encontraron logs para este modelo.')
        return None
    partes = []
    for i, archivo in enumerate(archivos):
        df = pd.read_csv(archivo)
        df['fragmento'] = i
        partes.append(df)
    log = pd.concat(partes, ignore_index=True)
    val_curve = log[['epoch', 'val_loss', 'fragmento']].dropna(subset=['val_loss'])
    val_curve = val_curve.sort_values(['epoch', 'fragmento']).drop_duplicates(subset='epoch', keep='last')
    print(val_curve.to_string(index=False))
    return val_curve

curva_t1_v2 = revisar_curva('tft_t1_v2')
curva_t2_v2 = revisar_curva('tft_t2_v2')
curva_t3_v2 = revisar_curva('tft_t3_v2')

tft_t1_v2: 1 fragmento(s) encontrados
 epoch  val_loss  fragmento
   0.0  1.159039          0
   1.0  1.142640          0
   2.0  1.185356          0
   3.0  1.174412          0
   4.0  1.150093          0
   5.0  1.150737          0
   6.0  1.155156          0
tft_t2_v2: 1 fragmento(s) encontrados
 epoch  val_loss  fragmento
   0.0  0.958449          0
   1.0  0.936700          0
   2.0  0.934961          0
   3.0  0.944209          0
   4.0  0.938937          0
   5.0  0.938361          0
   6.0  0.935841          0
   7.0  0.947529          0
tft_t3_v2: 1 fragmento(s) encontrados
 epoch  val_loss  fragmento
   0.0  0.937442          0
   1.0  0.949538          0
   2.0  0.936917          0
   3.0  0.937050          0
   4.0  0.937349          0
   5.0  0.934786          0
   6.0  0.942071          0
   7.0  0.951284          0
   8.0  0.941793          0
   9.0  0.948681          0
  10.0  0.943667          0
